# 오토인코더로 망가진 이미지 복원하기

잡음제거 오토인코더(Denoising Autoencoder)는 2008년 몬트리올 대학에서 발표한 논문 ["Extracting and Composing Robust Features with Denoising AutoEncoder"](http://www.cs.toronto.edu/~larocheh/publications/icml-2008-denoising-autoencoders.pdf)에서 처음 제안되었습니다.

앞서 오토인코더는 일종의 "압축"을 한다고 했습니다. 그리고 압축은 데이터의 특성에 중요도로 우선순위를 매기고 낮은 우선순위의 데이터를 버린다는 뜻이기도 합니다.

잡음제거 오토인코더의 아이디어는 중요한 특징을 추출하는 오토인코더의 특성을 이용하여 비교적 "덜 중요한 데이터"인 잡음을 버려 원래의 데이터를 복원한다는 것 입니다.

원래 배웠던 오토인코더와 큰 차이점은 없으며, 학습을 할때 입력에 잡음을 더하는 방식으로 복원 능력을 강화한 것이 핵심입니다. 앞서 다룬 코드와 동일하며 `add_noise()` 함수로 학습시 이미지에 노이즈를 더해주는 부분만 추가됐습니다.

In [ ]:
import torch
import torchvision
import torch.nn.functional as F
from torch import nn, optim
from torchvision import transforms, datasets

import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# 하이퍼파라미터
EPOCH = 10 # 전체 데이터셋 학습 횟수
BATCH_SIZE = 64 # 배치 크기 설정
USE_CUDA = torch.cuda.is_available() # GPU 사용 여부 체크
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 기기 할당
print("다음 기기로 학습합니다:", DEVICE) # 사용 기기 출력

In [ ]:
# 결과를 저장할 디렉토리 생성
if not os.path.exists('results'): # 결과 저장 폴더 체크
    os.makedirs('results') # 폴더 생성

In [ ]:
# Fashion MNIST 학습 데이터셋
trainset = datasets.FashionMNIST( # Fashion MNIST 데이터셋 로드
    root      = './.data/', # 저장 경로
    train     = True, # 학습용 선택
    download  = True, # 다운로드 여부
    transform = transforms.ToTensor() # 텐서 변환
)

train_loader = torch.utils.data.DataLoader( # 데이터 로더 설정
    dataset     = trainset, # 학습 데이터셋 지정
    batch_size  = BATCH_SIZE, # 배치 크기 설정
    shuffle     = True, # 섞기 활성화
    num_workers = 0 # 워커 수 설정
)

In [ ]:
class Autoencoder(nn.Module): # 오토인코더 클래스 정의
    def __init__(self): # 초기화 부분
        super(Autoencoder, self).__init__() # 부모 초기화

        self.encoder = nn.Sequential( # 인코더 레이어 구축
            nn.Linear(28*28, 128), # 평탄화된 입력 레이어
            nn.ReLU(), # 활성화 함수
            nn.Linear(128, 64), # 은닉층 1
            nn.ReLU(), # 활성화 함수
            nn.Linear(64, 12), # 은닉층 2
            nn.ReLU(), # 활성화 함수
            nn.Linear(12, 3), # 잠재 공간 차원 축소
        )
        self.decoder = nn.Sequential( # 디코더 레이어 구축
            nn.Linear(3, 12), # 복원 시작
            nn.ReLU(), # 활성화 함수
            nn.Linear(12, 64), # 확장 레이어 1
            nn.ReLU(), # 활성화 함수
            nn.Linear(64, 128), # 확장 레이어 2
            nn.ReLU(), # 활성화 함수
            nn.Linear(128, 28*28), # 원본 크기로 복원
            nn.Sigmoid(), # 출력값 정규화
        )

    def forward(self, x): # 순전파 흐름 정의
        encoded = self.encoder(x) # 차원 압축
        decoded = self.decoder(encoded) # 차원 복원
        return encoded, decoded # 압축값과 복원값 반환

In [ ]:
def add_noise(img): # 노이즈 추가 함수 정의
    noise = torch.randn(img.size()) * 0.2 # 가우시안 노이즈 생성
    noisy_img = img + noise # 이미지에 노이즈 합성
    return noisy_img # 오염된 이미지 반환

In [ ]:
autoencoder = Autoencoder().to(DEVICE) # 모델 생성 및 기기 할당
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.005) # 최적화 알고리즘
criterion = nn.MSELoss() # 평균 제곱 오차 손실 함수

In [ ]:
def train(autoencoder, train_loader): # 학습 함수
    autoencoder.train() # 학습 모드 설정
    avg_loss = 0 # 평균 손실 초기화
    for step, (x, label) in enumerate(train_loader): # 데이터 반복
        noisy_x = add_noise(x)  # 입력에 노이즈 더하기
        noisy_x = noisy_x.view(-1, 28*28).to(DEVICE) # 입력 평탄화
        y = x.view(-1, 28*28).to(DEVICE) # 입력 평탄화

        label = label.to(DEVICE) # 정답 이미지는 노이즈 없는 원본
        encoded, decoded = autoencoder(noisy_x) # 오염된 입력으로 학습

        loss = criterion(decoded, y) # 복원 오차 계산
        optimizer.zero_grad() # 기울기 초기화
        loss.backward() # 역전파
        optimizer.step() # 가중치 갱신
        
        avg_loss += loss.item() # 손실 누적
    return avg_loss / len(train_loader) # 배치당 평균 손실 반환

In [ ]:
for epoch in range(1, EPOCH+1): # 에포크 반복
    loss = train(autoencoder, train_loader) # 학습 수행
    print("[Epoch {}] loss:{}".format(epoch, loss)) # 진행 상황 출력
    # 이번 예제에선 학습시 시각화를 건너 뜁니다

# 이미지 복원 시각화 하기

In [ ]:
# 모델이 학습시 본적이 없는 데이터로 검증하기 위해 테스트 데이터셋을 가져옵니다.
testset = datasets.FashionMNIST( # 성능 검증용 테스트 데이터셋 로드
    root      = './.data/', 
    train     = False,
    download  = True,
    transform = transforms.ToTensor()
)

In [ ]:
# 테스트셋에서 이미지 한장을 가져옵니다.
sample_data = testset.data[0].view(-1, 28*28) # 테스트용 샘플 선택
sample_data = sample_data.type(torch.FloatTensor)/255. # 데이터 정규화

# 이미지를 add_noise로 오염시킨 후, 모델에 통과시킵니다.
original_x = sample_data[0] # 원본 이미지 추출
noisy_x = add_noise(original_x).to(DEVICE) # 노이즈 추가
_, recovered_x = autoencoder(noisy_x) # 모델을 통한 복원 수행

In [ ]:
f, a = plt.subplots(1, 3, figsize=(15, 15)) # 결과 비교 시각화

# 시각화를 위해 넘파이 행렬로 바꿔줍니다.
original_img = np.reshape(original_x.to("cpu").data.numpy(), (28, 28)) # 원본 변환
noisy_img = np.reshape(noisy_x.to("cpu").data.numpy(), (28, 28)) # 노이즈 이미지 변환
recovered_img = np.reshape(recovered_x.to("cpu").data.numpy(), (28, 28)) # 복원 이미지 변환

# 원본 사진
a[0].set_title('Original') # 제목 설정
a[0].imshow(original_img, cmap='gray') # 원본 출력

# 오염된 원본 사진
a[1].set_title('Noisy') # 제목 설정
a[1].imshow(noisy_img, cmap='gray') # 노이즈 출력

# 복원된 사진
a[2].set_title('Recovered') # 제목 설정
a[2].imshow(recovered_img, cmap='gray') # 복원 결과 출력

# 최종 결과 이미지 저장
plt.savefig('results/final_result.png', dpi=300, bbox_inches='tight') # 이미지 저장
plt.show() # 화면 표시

print("\n최종 결과 이미지가 'results/final_result.png'에 저장되었습니다.")